In [3]:
import cv2
import numpy as np
from collections import deque
from ultralytics import YOLO
import os

# ---------------- CONFIG ----------------
VIDEO_FOLDER = '../yolov8_model/videos'

SLOT_CAPACITIES = [3, 1, 2, 2, 1, 3]
NUM_SLOTS = len(SLOT_CAPACITIES)
NUM_REEFS = 2

HISTORY_LEN = 5
MIN_CONF = 0.5
TOP_ZONE_RATIO = 0.20

CONFIRM_FRAMES = 3   # frames to confirm coral placement
# ----------------------------------------

# States
EMPTY = 0
FILLING = 1
FILLED = 2

# Load model
model = YOLO('../best_yolov8_coral_reef/runs/detect/reef_coral/weights/best.pt')

# ---------------- HELPERS ----------------
def get_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2)/2, (y1 + y2)/2)

def assign_to_reef_and_slot(cx, cy, reefs):
    for r_idx, r in enumerate(reefs):
        rx1, ry1, rx2, ry2 = r
        rw, rh = rx2 - rx1, ry2 - ry1

        if (rx1 < cx < rx2) and (ry1 < cy < ry1 + rh*TOP_ZONE_RATIO):
            rel_x = cx - rx1
            slot_width = rw / NUM_SLOTS
            slot_margin = slot_width * 0.1
            slot = int((rel_x - slot_margin)/slot_width)
            slot = max(0, min(slot, NUM_SLOTS-1))
            return r_idx, slot
    return None, None

# ---------------- MAIN ----------------
video_files = [f for f in os.listdir(VIDEO_FOLDER) if f.lower().endswith(('.mp4','.avi'))]

for video_file in video_files:
    print(f"\nProcessing: {video_file}")

    cap = cv2.VideoCapture(os.path.join(VIDEO_FOLDER, video_file))

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cutoff_frame = total_frames - int(49*fps)

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    crop_h = int(height*(2/5))
    start_y = height - crop_h

    # --- STATE ---
    reef_grids = [[0]*NUM_SLOTS for _ in range(NUM_REEFS)]
    history = [[deque(maxlen=HISTORY_LEN) for _ in range(NUM_SLOTS)] for _ in range(NUM_REEFS)]

    # Markov state per slot
    states = [[EMPTY]*NUM_SLOTS for _ in range(NUM_REEFS)]
    timers = [[0]*NUM_SLOTS for _ in range(NUM_REEFS)]

    total_points = 0
    frame_idx = 0

    # --- LOOP ---
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= cutoff_frame:
            break

        cropped = frame[start_y:height, 0:width]
        results = model(cropped, conf=MIN_CONF, verbose=False)

        boxes = results[0].boxes.xyxy.cpu().numpy() if results[0].boxes else []
        clss  = results[0].boxes.cls.cpu().numpy().astype(int) if results[0].boxes else []
        confs = results[0].boxes.conf.cpu().numpy() if results[0].boxes else []

        reefs = [boxes[i] for i, c in enumerate(clss) if model.names[c]=='reef']
        reefs.sort(key=lambda x: x[0])

        if len(reefs) < NUM_REEFS:
            frame_idx += 1
            continue

        # 🔥 Keep only the expected number of reefs
        reefs = reefs[:NUM_REEFS]
        # --- COUNT RAW ---
        raw_counts = [[0]*NUM_SLOTS for _ in range(NUM_REEFS)]
        for i, c in enumerate(clss):
            if model.names[c] != 'coral' or confs[i] < MIN_CONF:
                continue

            cx, cy = get_center(boxes[i])
            r_idx, slot = assign_to_reef_and_slot(cx, cy, reefs)
            if (
                r_idx is not None and
                slot is not None and
                0 <= r_idx < NUM_REEFS and
                0 <= slot < NUM_SLOTS
            ):
                raw_counts[r_idx][slot] += 1
        # --- SMOOTH ---
        smoothed = [[0]*NUM_SLOTS for _ in range(NUM_REEFS)]
        for r in range(NUM_REEFS):
            for s in range(NUM_SLOTS):
                history[r][s].append(raw_counts[r][s])
                smoothed[r][s] = int(round(np.median(history[r][s])))
        # --- MARKOV STATE UPDATE (FIXED) ---
        DECAY = 1
        GROWTH = 1
        THRESHOLD = 4

        for r in range(NUM_REEFS):
            for s in range(NUM_SLOTS):

                curr = smoothed[r][s]
                placed = reef_grids[r][s]

                if placed >= SLOT_CAPACITIES[s]:
                    continue

                # Require stable detection
                stable = history[r][s].count(curr) >= 3

                if curr >= placed + 1 and stable:
                    timers[r][s] += GROWTH
                else:
                    timers[r][s] -= DECAY

                timers[r][s] = max(-THRESHOLD, timers[r][s])

                if timers[r][s] >= THRESHOLD:
                    reef_grids[r][s] += 1
                    total_points += 4

                    # 🔥 cooldown prevents recounting same coral
                    timers[r][s] = -THRESHOLD

    cap.release()

    # --- RESULTS ---
    print(f"Total points: {total_points}")
    for r in range(NUM_REEFS):
        print(f"Reef {r}: {reef_grids[r]}")


Processing: Qualification 72 - 2025 Iowa Regional.mp4
Total points: 24
Reef 0: [2, 1, 0, 0, 0, 3]
Reef 1: [0, 0, 0, 0, 0, 0]

Processing: Qualification 79 - 2025 FIRST Championship - Hopper Division presented by PwC.mp4
Total points: 48
Reef 0: [3, 1, 2, 2, 1, 3]
Reef 1: [0, 0, 0, 0, 0, 0]

Processing: Qualification 26 - 2025 Iowa Regional.mp4
Total points: 24
Reef 0: [0, 1, 0, 0, 0, 0]
Reef 1: [3, 1, 0, 0, 1, 0]

Processing: Qualification 15 - 2025 Iowa Regional.mp4
Total points: 72
Reef 0: [3, 1, 2, 2, 1, 3]
Reef 1: [2, 1, 1, 1, 0, 1]

Processing: Qualification 43 - 2025 FIRST Championship - Hopper Division presented by PwC.mp4
Total points: 88
Reef 0: [3, 1, 2, 1, 1, 3]
Reef 1: [2, 1, 2, 2, 1, 3]

Processing: Qualification 56 - 2025 Central Missouri Regional.mp4
Total points: 76
Reef 0: [3, 1, 2, 2, 1, 2]
Reef 1: [2, 1, 1, 1, 1, 2]

Processing: Final 1 - 2025 Central Missouri Regional.mp4
Total points: 84
Reef 0: [3, 1, 2, 2, 1, 0]
Reef 1: [3, 1, 2, 2, 1, 3]

Processing: Qualificat